# VishGym: closed multimodal QLoRA + group-relative self-play
This notebook trains only local, synthetic VishGym adapters. It never uploads an adapter, dataset, voice sample, transcript, or reward manifest. Run it from the repository root in a Colab GPU runtime.

In [ ]:
# First clone/upload VishGym and set this notebook's current directory to its repository root.
%pip install -q -e '.[training,audio,dev]'
!vishgym-train preflight

## Authenticate model downloads
Add `HF_TOKEN` in Colab Secrets (key icon), enable notebook access, and accept the Gemma model terms in Hugging Face before running this cell. Do not upload a local `.env` file.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
assert hf_token, 'Set HF_TOKEN in Colab Secrets and enable notebook access.'
login(token=hf_token, add_to_git_credential=False)
del hf_token

In [ ]:
# Closed-world, no-model smoke test. This remains intentionally CPU-safe.
from vishgym.arena.runner import run_local_episode
state, verdict = run_local_episode(seed=7)
assert verdict.terminal_outcome == 'safe_defense'
assert not hasattr(state, 'public_transcript')
print(verdict.model_dump())

## 1. Export a durable, audio-semantic warm-start dataset
This renders fixed built-in Qwen CustomVoice speakers into content-addressed local WAVs. It rejects reference audio and records no opponent transcript.

In [ ]:
!vishgym-train export-dataset --renderer qwen --output-dir artifacts/datasets/warm-start-qwen-v1 --seeds 7 11

In [ ]:
import json
from pathlib import Path

dataset_root = Path('artifacts/datasets/warm-start-qwen-v1')
dataset_manifest = json.loads((dataset_root / 'manifest.json').read_text())
assert dataset_manifest['synthetic_only']
assert dataset_manifest['contains_transcript'] is False
assert dataset_manifest['audio_training_eligible'] is True
DATASET_REVISION = dataset_manifest['revision']
print(DATASET_REVISION, dataset_manifest['example_count'])

## 2. Train initial role adapters
These are deliberately short smoke runs. Increase data diversity and steps only after reviewing the generated dataset and receipts.

In [ ]:
!vishgym-train warm-start --dataset-root artifacts/datasets/warm-start-qwen-v1 --role red --output-dir artifacts/adapters/red-sft-v1 --max-steps 60

In [ ]:
!vishgym-train warm-start --dataset-root artifacts/datasets/warm-start-qwen-v1 --role blue --output-dir artifacts/adapters/blue-sft-v1 --max-steps 60

## 3. Alternate closed group-relative rounds
Red first trains against the reviewed scripted Blue baseline. After Red's receipt is reviewed, Blue trains against the frozen local Red candidate. All rewards are terminal outcomes from the sandbox judge.

In [ ]:
!vishgym-train grpo --role red --initial-adapter-path artifacts/adapters/red-sft-v1/adapter --output-dir artifacts/adapters/red-grpo-round-1 --updates 3 --group-size 2

In [ ]:
!vishgym-train evaluate-red --red-adapter-path artifacts/adapters/red-grpo-round-1/adapter --adapter-revision red-grpo-round-1 --output-path artifacts/evaluations/red-grpo-round-1.json

In [ ]:
# Run only after the preceding Red receipt and review manifest have been human-reviewed.
!vishgym-train grpo --role blue --initial-adapter-path artifacts/adapters/blue-sft-v1/adapter --opponent-adapter-path artifacts/adapters/red-grpo-round-1/adapter --output-dir artifacts/adapters/blue-grpo-round-1 --updates 3 --group-size 2

In [ ]:
!vishgym-train evaluate-blue --blue-adapter-path artifacts/adapters/blue-grpo-round-1/adapter --red-adapter-path artifacts/adapters/red-grpo-round-1/adapter --dataset-revision {DATASET_REVISION} --adapter-revision blue-grpo-round-1 --output-path artifacts/evaluations/blue-grpo-round-1.json

## Review gate
The evaluation commands write actual held-out metrics and review-only manifests. They cannot publish, swap, or deploy any adapter. Retain frozen opponent adapters and repeat the Red → review → Blue sequence for rounds 2 and 3 with new held-out seeds/persona/timbre splits.